# ML-06 — Signal Audit: Do the Flags Hold?

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Ashritha-boop/fly_machine/blob/main/work/notebooks/w04_signal_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Distributions

*Look before deciding: distributions of your key fields. Note the heavy tails.*

Inspect the three declared model inputs before interpreting their binned associations. The report includes missingness, spread, and the observed April decline base rate; it does not establish that any feature causes decline.

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import os
import sys
import subprocess
from pathlib import Path

import numpy as np
import pandas as pd

try:
    import duckdb
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "duckdb"])
    import duckdb

HF_TOKEN = os.environ.get("HF_TOKEN")
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN")
    except Exception:
        HF_TOKEN = None

FEATURES = ["gsc_impressions_march", "gsc_avg_position_march", "content_age_days"]
TARGET = "is_declining_label"
con = duckdb.connect()
con.execute("INSTALL httpfs; LOAD httpfs;")
if HF_TOKEN:
    con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = "hf://datasets/FlyRank/internship-warehouse"
TABLES = {
    "dim_content": f"read_parquet('{REL}/dim_content.parquet')",
    "fact_daily": f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
}
data_origin = "warehouse"
try:
    con.sql(f"SELECT COUNT(*) FROM {TABLES['dim_content']}").fetchone()
    feature_sql = f"""
        WITH march AS (
            SELECT content_hash_id, client_hash_id,
                   SUM(gsc_impressions) AS gsc_impressions_march,
                   AVG(NULLIF(gsc_avg_position, 0)) AS gsc_avg_position_march
            FROM {TABLES['fact_daily']}
            WHERE report_date BETWEEN DATE '2026-03-01' AND DATE '2026-03-31'
            GROUP BY content_hash_id, client_hash_id
        ),
        april AS (
            SELECT content_hash_id, SUM(gsc_impressions) AS gsc_impressions_april
            FROM {TABLES['fact_daily']}
            WHERE report_date BETWEEN DATE '2026-04-01' AND DATE '2026-04-30'
            GROUP BY content_hash_id
        )
        SELECT m.content_hash_id, m.client_hash_id,
               m.gsc_impressions_march, m.gsc_avg_position_march,
               DATEDIFF('day', c.content_created_at, DATE '2026-03-31') AS content_age_days,
               a.gsc_impressions_april,
               CASE WHEN a.gsc_impressions_april < 0.80 * m.gsc_impressions_march THEN 1 ELSE 0 END AS is_declining_label
        FROM march m
        JOIN {TABLES['dim_content']} c USING (content_hash_id)
        JOIN april a USING (content_hash_id)
        WHERE m.gsc_impressions_march >= 10
    """
    df_features = con.sql(feature_sql).df()
except Exception as error:
    data_origin = "SIMULATED_STARTER_FALLBACK"
    print(f"Warehouse unavailable ({type(error).__name__}); using a clearly labeled local smoke-test frame.")
    local_csv = next((path for path in [
        Path("data/raw/content_refresh_anonymized.csv"),
        Path("../data/raw/content_refresh_anonymized.csv"),
    ] if path.exists()), None)
    if local_csv is None:
        local_csv = "https://raw.githubusercontent.com/Ashritha-boop/fly_machine/main/data/raw/content_refresh_anonymized.csv"
    raw = pd.read_csv(local_csv).head(5000).copy().reset_index(drop=True)
    np.random.seed(42)
    item_count = len(raw)
    march_impressions = np.clip(
        (raw["impressions_90d"] / 3 + np.random.normal(0, 100, item_count)).astype(int), 10, 50000
    )
    march_position = np.clip(raw["avg_position"] + np.random.normal(0, 0.5, item_count), 1.0, 99.0)
    decline_proxy = raw["trend_direction"].str.lower().eq("down").to_numpy()
    april_impressions = march_impressions.copy()
    april_impressions[decline_proxy] = (
        april_impressions[decline_proxy] * np.random.uniform(0.4, 0.75, decline_proxy.sum())
    ).astype(int)
    april_impressions[~decline_proxy] = (
        april_impressions[~decline_proxy] * np.random.uniform(0.9, 1.2, (~decline_proxy).sum())
    ).astype(int)
    march_total = (march_impressions // 31) * 31
    april_total = (march_impressions // 31 * (april_impressions / (march_impressions + 1))).astype(int) * 30
    df_features = pd.DataFrame({
        "content_hash_id": raw["content_id"],
        "client_hash_id": raw["client_id"],
        "gsc_impressions_march": march_total,
        "gsc_avg_position_march": march_position,
        "content_age_days": raw["content_age_days"],
        "gsc_impressions_april": april_total,
        "is_declining_label": (april_total < 0.80 * march_total).astype(int),
    })

df_features = df_features.dropna(subset=[TARGET]).copy()
print(f"Data source: {data_origin}")
print(f"Rows: {len(df_features):,}; observed April-decline base rate: {df_features[TARGET].mean():.3f}")
if data_origin != "warehouse":
    print("Fallback labels are simulated from the starter trend proxy; results are not warehouse signal evidence.")
distribution = df_features[FEATURES].describe(percentiles=[0.1, 0.5, 0.9]).T
distribution_missing = df_features[FEATURES].isna().mean().rename("missing_fraction")
distribution = distribution.join(distribution_missing)
print("Feature distributions:")
display(distribution.round(3))


Warehouse unavailable (BinderException); using a clearly labeled local smoke-test frame.
Data source: SIMULATED_STARTER_FALLBACK
Rows: 5,000; observed April-decline base rate: 0.490
Fallback labels are simulated from the starter trend proxy; results are not warehouse signal evidence.
Feature distributions:


,count,mean,std,min,10%,50%,90%,max,missing_fraction
gsc_impressions_march,5000.0,1722.868,4881.915,0.0,0.000,248.000,4123.000,49972.0,0.0
gsc_avg_position_march,5000.0,16.165,15.064,1.0,3.636,10.407,35.963,99.0,0.0
content_age_days,5000.0,255.951,134.312,90.0,104.000,236.000,463.000,564.0,0.0


## 2. Signal test #1 / #2 / #3 (verdict each)

*Three safe signals, each with a mini-test and a verdict: CONFIRMED / OPPOSITE / MIXED / FALSE.*

These are descriptive tests of the three candidate model features against the April decline label: (1) age bands, (2) March-impression quartiles, and (3) March-position bands. Each table reports the row count and observed decline fraction. Verdicts are directional only; no band was selected on a held-out set, and these summaries do not establish statistical or causal effects.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
age_edges = [-np.inf, 90, 180, 365, np.inf]
age_names = ["<90 days", "90-180 days", "180-365 days", "365+ days"]
df_features["age_band"] = pd.cut(df_features["content_age_days"], bins=age_edges, labels=age_names)
df_features["impression_quartile"] = pd.qcut(
    df_features["gsc_impressions_march"], q=4, duplicates="drop"
)
df_features["position_band"] = pd.cut(
    df_features["gsc_avg_position_march"].replace(0, np.nan),
    bins=[0, 3, 10, 20, 50, np.inf],
    labels=["1-3", "4-10", "11-20", "21-50", "50+"],
)


def signal_table(group_column):
    return (
        df_features.groupby(group_column, observed=False)[TARGET]
        .agg(rows="count", decline_rate="mean")
        .assign(decline_rate_pct=lambda table: 100 * table["decline_rate"])
        .reset_index()
    )


age_signal = signal_table("age_band")
impressions_signal = signal_table("impression_quartile")
position_signal = signal_table("position_band")
print("Signal 1 â€” content age bands vs next-month decline:")
display(age_signal.round(3))
print("Signal 2 â€” March impression quartiles vs next-month decline:")
display(impressions_signal.round(3))
print("Signal 3 â€” March average-position bands vs next-month decline:")
display(position_signal.round(3))
print(f"Rows without usable average-position band: {df_features['position_band'].isna().sum():,}")


Signal 1 â€” content age bands vs next-month decline:


,age_band,rows,decline_rate,decline_rate_pct
0,<90 days,83,0.482,48.193
1,90-180 days,2019,0.551,55.077
2,180-365 days,1820,0.466,46.648
3,365+ days,1078,0.415,41.466


Signal 2 â€” March impression quartiles vs next-month decline:


,impression_quartile,rows,decline_rate,decline_rate_pct
0,"(-0.001, 31.0]",1318,0.131,13.126
1,"(31.0, 248.0]",1271,0.676,67.585
2,"(248.0, 1147.0]",1168,0.628,62.842
3,"(1147.0, 49972.0]",1243,0.549,54.867


Signal 3 â€” March average-position bands vs next-month decline:


,position_band,rows,decline_rate,decline_rate_pct
0,1-3,388,0.286,28.608
1,4-10,2034,0.487,48.673
2,11-20,1171,0.549,54.910
3,21-50,1182,0.531,53.130
4,50+,225,0.338,33.778


Rows without usable average-position band: 0


## 3. The flag-linked test

*Pick a signal one of FlyRank's real flags relies on. Does the data support the rule's assumption?*

The staleness flag is `content_age_days >= 180`, which overlaps one of the Week-5 model inputs. We compare its observed April decline rate with the non-stale group and show counts and the overall base rate. This checks whether the flag is directionally associated in this frame; it is not an independent test because the cutoff is being assessed on the same cohort, and it does not measure whether refreshing a page changes its outcome.

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
df_features["stale_flag"] = df_features["content_age_days"] >= 180
staleness_table = (
    df_features.groupby("stale_flag")[TARGET]
    .agg(rows="count", decline_rate="mean")
    .assign(decline_rate_pct=lambda table: 100 * table["decline_rate"])
    .reset_index()
)
staleness_rates = staleness_table.set_index("stale_flag")["decline_rate"]
base_rate = df_features[TARGET].mean()
stale_difference = staleness_rates.get(True, np.nan) - staleness_rates.get(False, np.nan)
print("Staleness flag comparison (age >= 180 days):")
display(staleness_table.round(3))
print(f"Observed frame base rate: {base_rate:.3f}")
print(f"Observed stale-minus-non-stale rate difference: {stale_difference:+.3f}")
print("Verdict: directional only; same-frame threshold evaluation, not causal evidence.")

Staleness flag comparison (age >= 180 days):


,stale_flag,rows,decline_rate,decline_rate_pct
0,False,2057,0.550,55.032
1,True,2943,0.447,44.716


Observed frame base rate: 0.490
Observed stale-minus-non-stale rate difference: -0.103
Verdict: directional only; same-frame threshold evaluation, not causal evidence.


## 4. What this means in practice

*Two or three sentences: what a content team should take from this.*

Read the tables as observed associations among eligible items in one March-to-April frame. They can help prioritize what a person inspects, alongside the client-held-out model check in Week 6; they do not show that staleness, impressions, or rank caused the outcome. Keep the age/position flags as review context until the pattern is tested on unseen clients or later months, and use a controlled experiment to measure refresh impact.

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
print(f"Source: {data_origin}; rows: {len(df_features):,}; clients: {df_features['client_hash_id'].nunique():,}")
print(f"Observed decline base rate: {base_rate:.3f}")
print(f"Observed stale/non-stale decline-rate difference: {stale_difference:+.3f}")
print("Decision-support reading: use these fields to order human review, not to trigger an automatic refresh.")
if data_origin != "warehouse":
    print("This run is a synthetic smoke test only; do not quote its signal rates as warehouse findings.")

Source: SIMULATED_STARTER_FALLBACK; rows: 5,000; clients: 32
Observed decline base rate: 0.490
Observed stale/non-stale decline-rate difference: -0.103
Decision-support reading: use these fields to order human review, not to trigger an automatic refresh.
This run is a synthetic smoke test only; do not quote its signal rates as warehouse findings.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.